Cells below are all earlier attempts to visualize a topic model. A good deal of this code went into 5.3. There are multiple variants of the `drawviz()` function, some assisted by Chatgpt.

# Everything below is old - but potentially still useful.

# Visualize the Documents 1: Using MDS
While pyLDAvis is an excellent tool for exploring the topic/term aspect of a topic model (the words and their probabilities in each topic) it does not provide access to the document/topic aspect (the probability distribution of topics in each document). The visualization below plots all the documents according to their (cosine) distances (using Multi-Dimensional Scaling) in the Document/Term DataFrame. Each document (data point in the visualization) is colored according to the most prevalent topic and the size of the dot represents the probability of the most prevalent topic in that document.

Compute the distances between each of the documents. Use either the Document/Topic Dataframe or the Document/Term Dataframe (constructed below) to measure distance.

Since the data is already in list format, CountVectorizer does not need to preprocess or tokenize. The only way to prevent CountVectorizer from doing so is by creating dummy functions for the preprocessor and the tokenizer. The function DoNothing() simply returns the argument it receives.

In [ ]:
def DoNothing(x):
    return x

In [ ]:
df = topic_model['df']
texts = topic_model['texts']
cv = CountVectorizer(analyzer='word', preprocessor=DoNothing, tokenizer=DoNothing, 
                     token_pattern=None)
dtm = cv.fit_transform(texts)
dtm_df = pd.DataFrame(dtm.toarray(), columns = cv.get_feature_names_out(), index = df.index.values)
dtm_df.head()

In [ ]:
dist = squareform(pdist(dtm_df, 'cosine'))

Compute the position of each document using Multi-Dimensional Scaling. The variable `pos` holds the `x` and `y`  coordinates. Execution of the following cell may take several minutes.

In [ ]:
seed = 15
mds = MDS(
    n_components=2, 
    max_iter=3000,
    random_state=seed, 
    dissimilarity="precomputed", 
    n_jobs=1,
    normalized_stress='auto')
pos = mds.fit_transform(dist)

Create lists of x and y values (coordinates).

In [ ]:
mds_x = [x for x, y in pos]
mds_y = [y for x, y in pos]

Create lists of the most prevalent topic, the probability of the most prevalent topic, and the text name for each document. These lists are used in the tooltips of the Bokeh visualization.

In [ ]:
d_t_df = topic_model['d_t_df']
prevalent_topic = d_t_df.idxmax(axis=1)
probability = d_t_df.max(axis=1)
designation = list(df['designation'])

# Main Visualization Function
Interactive features in Bokeh, such as a drop-down menu, use a callback function that is activated when a certain event takes place. This event can be a mouse movement, a click, or a change in the drop-down menu. Custom callback functions need to be written in JavaScript.

Draw the visualization. The visualization provides various tools for further exploration:
- tooltips (provides topic, probability, text name and URL)
- box zoom
- wheel zoom
- pan
- reset
- link to document edition
- save the visualization

In addition, the visualization has a drop-down menu that allow the user to select two topics.
This function was written with feedback from ChatGPT.

In [ ]:
def drawviz(title, outputfile):
    colormap = {
        str(i): color for i, color in enumerate([
            'grey', 'orange', 'olive', 'firebrick', 'gold', 'red', 'fuchsia', 'green',
            'blue', 'purple', 'aqua', 'yellow', 'indigo', 'blueviolet', 'beige', 'navy',
            'chocolate', 'azure', 'coral', 'crimson', 'darkblue', 'darkkhaki', 
            'darkseagreen', 'darkturquoise', 'deeppink', 'black'
        ])
    }

    colormap_source = ColumnDataSource(data=dict(
        topic=list(colormap.keys()),
        color=list(colormap.values())
    ))

    d_mds = dict(
        x=mds_x,
        y=mds_y,
        id_text=list(df.id_text),
        size=probability / max(probability) * 15,
        probability=probability,
        topic=[str(n) for n in prevalent_topic],  # ensure string topics
        color=[colormap[str(n)] for n in prevalent_topic],
        alpha=[0.5] * len(mds_x),
        designation=designation
    )

    source_mds = ColumnDataSource(data=d_mds)

    p = figure(width=1000, height=1000,
               tools="tap,pan,wheel_zoom,box_zoom,reset,save", 
               title=title)

    p.add_tools(HoverTool(tooltips=[
        ("url", "http://oracc.org/" + "@id_text"),
        ("topic, probability", "@topic, @probability"),
        ("designation", "@designation")
    ]))

    p.circle('x', 'y', color='color', fill_alpha='alpha', size='size', source=source_mds)
    renderers = []
    for topic, color in colormap.items():
        r = p.circle(
            [], [],  # empty glyph
            color=color,
            size=10,
            legend_label=f"Topic {topic}"
            )
        renderers.append(r)

    p.legend.title = "Topics"
    p.legend.location = "top_right"
    p.legend.click_policy = "hide"
    p.axis.visible = False

    # MultiChoice selector instead of sliders
    topic_selector = MultiChoice(title="Select Topics",
                                 options=[str(i+1) for i in range(ntopics)],
                                 value=[])

    callback = CustomJS(args=dict(
        source=source_mds,
        selector=topic_selector,
        colormap=colormap_source
    ), code="""
        const selected = selector.value;
        const data = source.data;
        const topics = data['topic'];
        const alpha = data['alpha'];
        const colors = data['color'];
        const cmap_data = colormap.data;

        let cmap = {};
        for (let i = 0; i < cmap_data['topic'].length; i++) {
            cmap[cmap_data['topic'][i]] = cmap_data['color'][i];
        }

        for (let i = 0; i < topics.length; i++) {
            const currentTopic = topics[i];
            const highlight = selected.length === 0 || selected.includes(currentTopic);
            alpha[i] = highlight ? 0.5 : 0.1;
            colors[i] = highlight ? cmap[currentTopic] : 'grey';
        }

        source.change.emit();
    """)

    topic_selector.js_on_change('value', callback)

    # Tap tool
    taptool = p.select(type=TapTool)
    taptool.callback = OpenURL(url="http://oracc.museum.upenn.edu/@id_text")

    instructions = [
        "Highlight one or more topics using the dropdown. If none are selected, all are shown.",
        "Hover over a data point for more info. Click to view the document edition.",
        "Use the toolbar to zoom, pan, reset, or save as .png."
    ]
    for line in instructions:
        p.add_layout(Title(text=line), 'below')

    layout = column(topic_selector, p)
    reset_output()
    output_file(outputfile)
    save(layout)
    show(layout)


In [ ]:
drawviz('Visualize with MDS', 'vis/lda_mds.html')

## Alternative: plotting based on Document/Topic table
The following visualization uses the same approach, but takes the document/topic table as the basis for distance measurements. Documents that share approximately the same distribution of topics will be plotted in the same region. Since the sum of each row in the document/topic table is 1 the distance matrix is computed with euclidean distance (not cosine).

In [ ]:
dist_dt = squareform(pdist(d_t_df))

In [ ]:
mds = MDS(n_components=2, max_iter=3000,
       random_state=seed, dissimilarity="precomputed", n_jobs=1)
pos = mds.fit_transform(dist_dt)

In [ ]:
drawviz(d_mds2, 'Visualize with MDS, version 2', 'vis/lda_mds2.html')

In [ ]:
d_mds2 = d_mds.copy() # the data source is the same as for the previous visualization, except for the x and y coordinates.
d_mds2['x'] = [x for x, y in pos]
d_mds2['y'] = [y for x, y in pos]

In [ ]:
reset_output()
output_notebook()
title = "Projection with MDS, based on Document/Topic distribution. Size of the circle represents prevalence of the topic."
outputfile = 'vis/mds2.html'
drawviz(d_mds2, title, outputfile)

# Visualize the Documents 2: Using TSNE

# TSNE based on Document/Term Matrix (Cosine distance)

Cosine distances have been computed earlier; the matrix is stored in the variable `dist`.

In [ ]:
X = dist
tsne = TSNE(n_components = 2, random_state=0, metric="precomputed", init="random")
X_tsne = tsne.fit_transform(X)

In [ ]:
d_tsne = d_mds.copy() # the data source is the same as for the previous visualization, except for the x and y coordinates.
d_tsne['x'] = [x for x, y in X_tsne]
d_tsne['y'] = [y for x, y in X_tsne]

In [ ]:
title = "Projection with tSNE. Size of the circle represents prevalence of the topic."
outputfile = 'vis/tsne1.html'
drawviz(d_tsne, outputfile)

# TSNE based on Document/Topic Matrix

In [ ]:
X = dist_dt
tsne = TSNE(n_components = 2, random_state=0, metric="precomputed")
X_tsne = tsne.fit_transform(X)

In [ ]:
colormap = {
        str(i): color for i, color in enumerate([
            'grey', 'orange', 'olive', 'firebrick', 'gold', 'red', 'fuchsia', 'green',
            'blue', 'purple', 'aqua', 'yellow', 'indigo', 'blueviolet', 'beige', 'navy',
            'chocolate', 'azure', 'coral', 'crimson', 'darkblue', 'darkkhaki', 
            'darkseagreen', 'darkturquoise', 'deeppink', 'black'
        ])
    }


In [ ]:
d_mds = dict(
        x=mds_x,
        y=mds_y,
        id_text=list(df.id_text),
        size=probability / max(probability) * 15,
        probability=probability,
        topic=[str(n) for n in prevalent_topic],  # ensure string topics
        color=[colormap[str(n)] for n in prevalent_topic],
        alpha=[0.5] * len(mds_x),
        designation=designation
    )

In [ ]:
d_tsne2 = d_mds.copy() # the data source is the same as for the previous visualization, except for the x and y coordinates.
d_tsne2['x'] = [x for x, y in X_tsne]
d_tsne2['y'] = [y for x, y in X_tsne]

In [ ]:
title = "Projection with tSNE, based on Document/Topic distribution. Size of the circle represents prevalence of the topic."
outputfile = 'vis/tsne2.html'
drawviz(d_tsne2, outputfile)

Suggested by chatGPT

In [ ]:
from sklearn.manifold import TSNE

seed = 15
tsne = TSNE(
    n_components=2,
    perplexity=30,
    metric='precomputed',  # Use if `dist` is a distance matrix
    random_state=seed,
    init='random',         # Or 'pca' if preferred
    max_iter=1000
)

pos = tsne.fit_transform(dist)


In [ ]:
tsne_x = pos[:, 0]
tsne_y = pos[:, 1]

In [ ]:
def drawviz(title, outputfile):
    colormap = {
        str(i): color for i, color in enumerate([
            'grey', 'orange', 'olive', 'firebrick', 'gold', 'red', 'fuchsia', 'green',
            'blue', 'purple', 'aqua', 'yellow', 'indigo', 'blueviolet', 'beige', 'navy',
            'chocolate', 'azure', 'coral', 'crimson', 'darkblue', 'darkkhaki', 
            'darkseagreen', 'darkturquoise', 'deeppink', 'black'
        ])
    }

    colormap_source = ColumnDataSource(data=dict(
        topic=list(colormap.keys()),
        color=list(colormap.values())
    ))

    d_tsne = dict(
    x=tsne_x,
    y=tsne_y,
    id_text=list(df.id_text),
    size=probability / max(probability) * 15,
    probability=probability,
    topic=[str(n) for n in prevalent_topic],
    color=[colormap[str(n)] for n in prevalent_topic],
    alpha=[0.5] * len(tsne_x),
    designation=designation
    )


    source_tsne = ColumnDataSource(data=d_tsne)

    p = figure(width=1000, height=1000,
               tools="tap,pan,wheel_zoom,box_zoom,reset,save", 
               title=title)

    p.add_tools(HoverTool(tooltips=[
        ("url", "http://oracc.org/" + "@id_text"),
        ("topic, probability", "@topic, @probability"),
        ("designation", "@designation")
    ]))

    p.circle('x', 'y', color='color', fill_alpha='alpha', size='size', source=source_tsne)
    p.axis.visible = False

    # MultiChoice selector instead of sliders
    topic_selector = MultiChoice(title="Select Topics",
                                 options=[str(i+1) for i in range(ntopics)],
                                 value=[])

    callback = CustomJS(args=dict(
        source=source_tsne,
        selector=topic_selector,
        colormap=colormap_source
    ), code="""
        const selected = selector.value;
        const data = source.data;
        const topics = data['topic'];
        const alpha = data['alpha'];
        const colors = data['color'];
        const cmap_data = colormap.data;

        let cmap = {};
        for (let i = 0; i < cmap_data['topic'].length; i++) {
            cmap[cmap_data['topic'][i]] = cmap_data['color'][i];
        }

        for (let i = 0; i < topics.length; i++) {
            const currentTopic = topics[i];
            const highlight = selected.length === 0 || selected.includes(currentTopic);
            alpha[i] = highlight ? 0.5 : 0.1;
            colors[i] = highlight ? cmap[currentTopic] : 'grey';
        }

        source.change.emit();
    """)

    topic_selector.js_on_change('value', callback)

    # Tap tool
    taptool = p.select(type=TapTool)
    taptool.callback = OpenURL(url="http://oracc.museum.upenn.edu/@id_text")

    instructions = [
        "Highlight one or more topics using the dropdown. If none are selected, all are shown.",
        "Hover over a data point for more info. Click to view the document edition.",
        "Use the toolbar to zoom, pan, reset, or save as .png."
    ]
    for line in instructions:
        p.add_layout(Title(text=line), 'below')

    layout = column(topic_selector, p)
    reset_output()
    output_file(outputfile)
    save(layout)
    show(layout)

In [ ]:
drawviz('Visualize with tsne', 'vis/lda_tsne.html')

In [ ]:
import umap

seed = 15
reducer = umap.UMAP(
    n_components=2,
    metric='precomputed',   # Use this if 'dist' is a distance matrix
    random_state=seed
)

In [ ]:
pos = reducer.fit_transform(dist)  # dist = your precomputed distance matrix

In [ ]:
umap_x = pos[:, 0]
umap_y = pos[:, 1]

In [ ]:
def drawviz(title, outputfile):
    colormap = {
        str(i): color for i, color in enumerate([
            'grey', 'orange', 'olive', 'firebrick', 'gold', 'red', 'fuchsia', 'green',
            'blue', 'purple', 'aqua', 'yellow', 'indigo', 'blueviolet', 'beige', 'navy',
            'chocolate', 'azure', 'coral', 'crimson', 'darkblue', 'darkkhaki', 
            'darkseagreen', 'darkturquoise', 'deeppink', 'black'
        ])
    }

    colormap_source = ColumnDataSource(data=dict(
        topic=list(colormap.keys()),
        color=list(colormap.values())
    ))

    d_umap = dict(
    x=umap_x,
    y=umap_y,
    id_text=list(df.id_text),
    size=probability / max(probability) * 15,
    probability=probability,
    topic=[str(n) for n in prevalent_topic],
    color=[colormap[str(n)] for n in prevalent_topic],
    alpha=[0.5] * len(umap_x),
    designation=designation
    )


    source_umap = ColumnDataSource(data=d_umap)

    p = figure(width=1000, height=1000,
               tools="tap,pan,wheel_zoom,box_zoom,reset,save", 
               title=title)

    p.add_tools(HoverTool(tooltips=[
        ("url", "http://oracc.org/" + "@id_text"),
        ("topic, probability", "@topic, @probability"),
        ("designation", "@designation")
    ]))

    p.circle('x', 'y', color='color', fill_alpha='alpha', size='size', source=source_umap)
    p.axis.visible = False

    # MultiChoice selector instead of sliders
    topic_selector = MultiChoice(title="Select Topics",
                                 options=[str(i+1) for i in range(ntopics)],
                                 value=[])

    callback = CustomJS(args=dict(
        source=source_umap,
        selector=topic_selector,
        colormap=colormap_source
    ), code="""
        const selected = selector.value;
        const data = source.umap_data;
        const topics = data['topic'];
        const alpha = data['alpha'];
        const colors = data['color'];
        const cmap_data = colormap.data;

        let cmap = {};
        for (let i = 0; i < cmap_data['topic'].length; i++) {
            cmap[cmap_data['topic'][i]] = cmap_data['color'][i];
        }

        for (let i = 0; i < topics.length; i++) {
            const currentTopic = topics[i];
            const highlight = selected.length === 0 || selected.includes(currentTopic);
            alpha[i] = highlight ? 0.5 : 0.1;
            colors[i] = highlight ? cmap[currentTopic] : 'grey';
        }

        source.change.emit();
    """)

    topic_selector.js_on_change('value', callback)

    # Tap tool
    taptool = p.select(type=TapTool)
    taptool.callback = OpenURL(url="http://oracc.museum.upenn.edu/@id_text")

    instructions = [
        "Highlight one or more topics using the dropdown. If none are selected, all are shown.",
        "Hover over a data point for more info. Click to view the document edition.",
        "Use the toolbar to zoom, pan, reset, or save as .png."
    ]
    for line in instructions:
        p.add_layout(Title(text=line), 'below')

    layout = column(topic_selector, p)
    reset_output()
    output_file(outputfile)
    save(layout)
    show(layout)

In [ ]:
from bokeh.models import (
    ColumnDataSource, HoverTool, TapTool, OpenURL, CustomJS, MultiChoice, Title
)
from bokeh.plotting import figure, output_file, save, show, reset_output
from bokeh.layouts import column

In [ ]:
def drawviz(title, outputfile):
    colormap = {
        str(i): color for i, color in enumerate([
            'grey', 'orange', 'olive', 'firebrick', 'gold', 'red', 'fuchsia', 'green',
            'blue', 'purple', 'aqua', 'yellow', 'indigo', 'blueviolet', 'beige', 'navy',
            'chocolate', 'azure', 'coral', 'crimson', 'darkblue', 'darkkhaki', 
            'darkseagreen', 'darkturquoise', 'deeppink', 'black'
        ])
    }

    colormap_source = ColumnDataSource(data=dict(
        topic=list(colormap.keys()),
        color=list(colormap.values())
    ))

    d_umap = dict(
    x=umap_x,
    y=umap_y,
    id_text=list(df.id_text),
    size=probability / max(probability) * 15,
    probability=probability,
    topic=[str(n) for n in prevalent_topic],
    color=[colormap[str(n)] for n in prevalent_topic],
    alpha=[0.5] * len(umap_x),
    designation=designation
    )


    source_umap = ColumnDataSource(data=d_umap)

    p = figure(width=1000, height=1000,
               tools="tap,pan,wheel_zoom,box_zoom,reset,save", 
               title=title)

    p.add_tools(HoverTool(tooltips=[
        ("url", "http://oracc.org/" + "@id_text"),
        ("topic, probability", "@topic, @probability"),
        ("designation", "@designation")
    ]))

    p.circle('x', 'y', color='color', fill_alpha='alpha', size='size', source=source_umap)
    p.axis.visible = False

    # MultiChoice selector instead of sliders
    topic_selector = MultiChoice(title="Select Topics",
                                 options=[str(i+1) for i in range(ntopics)],
                                 value=[])

    callback = CustomJS(args=dict(
        source=source_umap,
        selector=topic_selector,
        colormap=colormap_source
    ), code="""
        const selected = selector.value;
        const data = source.data;
        const topics = data['topic'];
        const alpha = data['alpha'];
        const colors = data['color'];
        const cmap_data = colormap.data;

        let cmap = {};
        for (let i = 0; i < cmap_data['topic'].length; i++) {
            cmap[cmap_data['topic'][i]] = cmap_data['color'][i];
        }

        for (let i = 0; i < topics.length; i++) {
            const currentTopic = topics[i];
            const highlight = selected.length === 0 || selected.includes(currentTopic);
            alpha[i] = highlight ? 0.5 : 0.1;
            colors[i] = highlight ? cmap[currentTopic] : 'grey';
        }

        source.change.emit();
    """)

    topic_selector.js_on_change('value', callback)

    # Tap tool
    taptool = p.select(type=TapTool)
    taptool.callback = OpenURL(url="http://oracc.museum.upenn.edu/@id_text")

    instructions = [
        "Highlight one or more topics using the dropdown. If none are selected, all are shown.",
        "Hover over a data point for more info. Click to view the document edition.",
        "Use the toolbar to zoom, pan, reset, or save as .png."
    ]
    for line in instructions:
        p.add_layout(Title(text=line), 'below')

    layout = column(topic_selector, p)
    reset_output()
    output_file(outputfile)
    save(layout)
    show(layout)

In [ ]:
drawviz("UMAP Visualization of Topics", "vis/topic_umap.html")

In [ ]:
from sklearn.manifold import MDS, TSNE
import umap
from bokeh.models import (
    ColumnDataSource, HoverTool, TapTool, OpenURL, CustomJS, MultiChoice, Title
)
from bokeh.plotting import figure, output_file, save, show, reset_output
from bokeh.layouts import column

def compute_embedding(dist, method='umap', seed=15):
    if method == 'umap':
        reducer = umap.UMAP(n_components=2, metric='precomputed', random_state=seed)
        pos = reducer.fit_transform(dist)
    elif method == 'mds':
        mds = MDS(
            n_components=2,
            max_iter=3000,
            random_state=seed,
            dissimilarity='precomputed',
            n_jobs=1,
            normalized_stress='auto'
        )
        pos = mds.fit_transform(dist)
    elif method == 'tsne':
        tsne = TSNE(
            n_components=2,
            perplexity=30,
            metric='precomputed',
            random_state=seed,
            init='random',
            n_iter=1000
        )
        pos = tsne.fit_transform(dist)
    else:
        raise ValueError(f"Unknown method '{method}'. Choose 'umap', 'mds', or 'tsne'.")
    return pos

def drawviz(title, outputfile, dist, method='umap'):
    # Compute embedding coordinates based on chosen method
    pos = compute_embedding(dist, method=method)
    x = pos[:, 0]
    y = pos[:, 1]

    colormap = {
        str(i): color for i, color in enumerate([
            'grey', 'orange', 'olive', 'firebrick', 'gold', 'red', 'fuchsia', 'green',
            'blue', 'purple', 'aqua', 'yellow', 'indigo', 'blueviolet', 'beige', 'navy',
            'chocolate', 'azure', 'coral', 'crimson', 'darkblue', 'darkkhaki',
            'darkseagreen', 'darkturquoise', 'deeppink', 'black'
        ])
    }

    colormap_source = ColumnDataSource(data=dict(
        topic=list(colormap.keys()),
        color=list(colormap.values())
    ))

    d = dict(
        x=x,
        y=y,
        id_text=list(df.id_text),
        size=probability / max(probability) * 15,
        probability=probability,
        topic=[str(n) for n in prevalent_topic],
        color=[colormap[str(n)] for n in prevalent_topic],
        alpha=[0.5] * len(x),
        designation=designation
    )

    source = ColumnDataSource(data=d)

    p = figure(width=1000, height=1000,
               tools="tap,pan,wheel_zoom,box_zoom,reset,save",
               title=title)

    p.add_tools(HoverTool(tooltips=[
        ("url", "http://oracc.org/" + "@id_text"),
        ("topic, probability", "@topic, @probability"),
        ("designation", "@designation")
    ]))

    p.circle('x', 'y', color='color', fill_alpha='alpha', size='size', source=source)
    p.axis.visible = False

    topic_selector = MultiChoice(title="Select Topics",
                                 options=[str(i) for i in range(ntopics)],
                                 value=[])

    callback = CustomJS(args=dict(
        source=source,
        selector=topic_selector,
        colormap=colormap_source
    ), code="""
        const selected = selector.value;
        const data = source.data;
        const topics = data['topic'];
        const alpha = data['alpha'];
        const colors = data['color'];
        const cmap_data = colormap.data;

        let cmap = {};
        for (let i = 0; i < cmap_data['topic'].length; i++) {
            cmap[cmap_data['topic'][i]] = cmap_data['color'][i];
        }

        for (let i = 0; i < topics.length; i++) {
            const currentTopic = topics[i];
            const highlight = selected.length === 0 || selected.includes(currentTopic);
            alpha[i] = highlight ? 0.5 : 0.1;
            colors[i] = highlight ? cmap[currentTopic] : 'grey';
        }

        source.change.emit();
    """)

    topic_selector.js_on_change('value', callback)

    taptool = p.select(type=TapTool)
    taptool.callback = OpenURL(url="http://oracc.museum.upenn.edu/@id_text")

    instructions = [
        "Highlight one or more topics using the dropdown. If none are selected, all are shown.",
        "Hover over a data point for more info. Click to view the document edition.",
        "Use the toolbar to zoom, pan, reset, or save as .png."
    ]
    for line in instructions:
        p.add_layout(Title(text=line), 'below')

    layout = column(topic_selector, p)
    reset_output()
    output_file(outputfile)
    save(layout)
    show(layout)


In [ ]:
drawviz("Topic Visualization (UMAP)", "output_umap.html", dist, method='umap')   # default
drawviz("Topic Visualization (MDS)", "output_mds.html", dist, method='mds')
drawviz("Topic Visualization (t-SNE)", "output_tsne.html", dist, method='tsne')

In [ ]:
drawviz("Topic Visualization (MDS)", "output_mds.html", dist, method='mds')

In [ ]:
drawviz("Topic Visualization (TSNE)", "vis/tsne.html", dist, method='tsne')